# Synthea Coherent Data Set Tutorial

The **Synthea Coherent Data Set** is a unique multimodal synthetic healthcare dataset combining:

- **EHR (FHIR)**: Patient records with demographics, conditions, medications, encounters
- **Imaging (DICOM)**: Synthetic brain MRI scans
- **Genomics (CSV)**: DNA test results (`*_dna.csv` files)
- **Clinical Notes**: SOAP-style documentation

All modalities are linked via patient UUIDs.

**References**: [Paper](https://www.mdpi.com/2079-9292/11/8/1199) | [AWS Open Data](https://registry.opendata.aws/synthea-coherent-data/) | S3: `s3://synthea-open-data/coherent/`

In [1]:
%load_ext autoreload
%autoreload 2

import synthlab as sl


  ░██████╗██╗░░░██╗███╗░░██╗████████╗██╗░░██╗  ██╗░░░░░░█████╗░██████╗░
  ██╔════╝╚██╗░██╔╝████╗░██║╚══██╔══╝██║░░██║  ██║░░░░░██╔══██╗██╔══██╗
  ╚█████╗░░╚████╔╝░██╔██╗██║░░░██║░░░███████║  ██║░░░░░███████║██████╦╝
  ░╚═══██╗░░╚██╔╝░░██║╚████║░░░██║░░░██╔══██║  ██║░░░░░██╔══██║██╔══██╗
  ██████╔╝░░░██║░░░██║░╚███║░░░██║░░░██║░░██║  ███████╗██║░░██║██████╦╝
  ╚═════╝░░░░╚═╝░░░╚═╝░░╚══╝░░░╚═╝░░░╚═╝░░╚═╝  ╚══════╝╚═╝░░╚═╝╚═════╝░
        ▌║█║▌│║▌│║▌║▌█║▌║█║▌│║▌│║▌║▌█║▌║█║▌│║▌│║▌║▌█║▌│║▌│║▌║▌█║
  ════════════════════════════════════════════════════════════════════
  Synthetic Healthcare Data Toolkit
  ────────────────────────────────────────────────────────────────────
  ◈ EHR        Synthea FHIR + OMOP
  ◈ Genomics   HAPNEST synthetic genotypes
  ◈ Imaging    Medical imaging catalog
  ◈ Coherent   Multimodal EHR + Imaging + Genomics
  ════════════════════════════════════════════════════════════════════

  Version: 0.2.0
  Cache:   /home/schilder/.cache/synthlab



## 1. Dataset Overview

In [39]:
# Print detailed information about the dataset
sl.print_coherent_info()


Synthea Coherent Data Set

Multimodal synthetic healthcare data combining EHR, imaging, genomics, notes, and physiological data

Source: s3://synthea-open-data/coherent/
License: CC BY 4.0
Paper: https://www.mdpi.com/2079-9292/11/8/1199

----------------------------------------------------------------------
Components:
----------------------------------------------------------------------

  FHIR
    FHIR R4 patient records (JSON bundles)
    Format: JSON (FHIR R4)
    Contents: Demographics, conditions, medications, encounters, observations, etc.

  DICOM
    MRI brain imaging (DICOM files)
    Format: DICOM
    Contents: Synthetic brain MRI scans linked to patients

  GENOMICS
    DNA/genomics data (CSV format with genetic variants)
    Format: CSV
    Contents: Synthetic genetic data for patients (DNA test results)

  CSV
    Tabular data exports
    Format: CSV
    Contents: Structured tabular data exports

----------------------------------------------------------------------
Req

## 2. Downloading Data

Download specific components or limit patients for testing. Full dataset is several GB.

In [ ]:
# List available files before downloading
fhir_files = sl.list_coherent_files(component='fhir', max_files=5)
print("Sample FHIR files available:")
for f in fhir_files:
    print(f"  {f['key'].split('/')[-1]} ({f['size_mb']:.2f} MB)")

In [ ]:
# Download FHIR + Genomics (uncomment to run)
# sl.download_coherent_dataset(components=['fhir', 'genomics'], max_patients=10)

print(f"Cache directory: {sl.get_coherent_cache_dir()}")

## 3. Loading Multimodal Data

Use `load_multimodal_dataset()` to load all available modalities with a single function call.

In [ ]:
# Load the multimodal dataset
dataset = sl.load_multimodal_dataset(max_patients=10)

# Print summary
dataset.print_summary()

In [ ]:
# Iterate over patients
for patient in dataset[:3]:
    demographics = patient.get_demographics()
    if demographics:
        print(f"\n{demographics['name']} ({patient.patient_id[:8]}...)")
        print(f"  Modalities: {patient.modalities}")
        print(f"  Conditions: {len(patient.get_conditions())}, Medications: {len(patient.get_medications())}")

## 4. Filtering and Querying

In [ ]:
# Filter by modality
print(f"Patients with FHIR: {len(dataset.with_modality('fhir'))}")
print(f"Patients with genomics: {len(dataset.with_modality('genomics'))}")
print(f"Patients with both: {len(dataset.with_modalities(['fhir', 'genomics']))}")

# Filter by clinical criteria
print(f"\nPatients with diabetes: {len(dataset.with_condition('diabetes'))}")
print(f"Female patients: {len(dataset.with_gender('female'))}")

In [ ]:
# Get all conditions as DataFrame
conditions_df = dataset.conditions_dataframe()
print(f"Total conditions: {len(conditions_df)}")
print(conditions_df.head())

## 5. Accessing Specific Modalities

In [ ]:
# Access genomics data for a patient
patient = dataset.with_modality('genomics')[0] if len(dataset.with_modality('genomics')) > 0 else None

if patient and patient.genomics is not None:
    print(f"Patient: {patient.name}")
    print(f"Genomics shape: {patient.genomics.shape}")
    print(patient.genomics.head())
else:
    print("No genomics data available")

In [ ]:
# Access DICOM imaging for a patient
patient = dataset.with_modality('dicom')[0] if len(dataset.with_modality('dicom')) > 0 else None

if patient and patient.dicom_paths:
    import matplotlib.pyplot as plt
    
    ds = patient.load_dicom(0)
    print(f"Patient: {patient.name}")
    print(f"Modality: {ds.get('Modality', 'Unknown')}, Shape: {ds.pixel_array.shape}")
    
    # Display middle slice if 3D
    img = ds.pixel_array
    if img.ndim == 3:
        img = img[img.shape[0] // 2]
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.show()
else:
    print("No DICOM data available")

## Summary

```python
import synthlab as sl

# Load everything with one call
dataset = sl.load_multimodal_dataset(max_patients=100)

# Filter and query
diabetics = dataset.with_condition('diabetes')
multimodal = dataset.with_modalities(['fhir', 'genomics'])

# Access patient data
for patient in dataset:
    demographics = patient.get_demographics()
    conditions = patient.get_conditions()
    genomics_df = patient.genomics  # Lazy loaded

# Export to DataFrames
conditions_df = dataset.conditions_dataframe()
medications_df = dataset.medications_dataframe()
```

**Resources**: [Paper](https://www.mdpi.com/2079-9292/11/8/1199) | [AWS Open Data](https://registry.opendata.aws/synthea-coherent-data/) | [FHIR R4](https://hl7.org/fhir/R4/)